In [ ]:
# copy from FINDER_percolation GraphDQN.pyx
# it is a class function

import networkx as nx
import random
import numpy as np
import TreeBreaker

def GetCoreDQNSolution(self, model_file, data_test_name, data_save_path, STEPRATIO):
    inner_env = mvc_env.py_MvcEnv(NUM_MAX)
    data_test_path = '../data/real/'
    ## begin computing...
    self.LoadModel(model_file)

    for dataName in data_test_name:
        print ('Calculating data: %s'%dataName)
        sol_decycle = []
        data_test = data_test_path + dataName + '.txt'
        data_save = data_save_path + dataName + '.txt'

        f_out = open(data_save,'w')
        ##read graph and do idMapping
        g_original = nx.read_edgelist(data_test)
        g_kcore, dmax, H, max_score_dict, score_dict, N0, N = self.preprocess(2, g_original)
        ## get decycle set
        done = False
        iteration = 0
        while N > 0 and done == False:
            ######################## GraphDQN dismantle ##############
            print ('Iteration: %d'%iteration)
            iteration += 1
            #do node id mapping
            Node2ID = {}
            ID2Node = {}
            numrealnodes = 0
            for node in g_kcore.nodes():
                ID2Node[numrealnodes] = node
                Node2ID[node] = numrealnodes
                numrealnodes += 1
            #get kcore of graph
            #get inner graph with new idMapping
            g_kcore_inner = self.GenNetworkWithIDMapping(g_kcore, Node2ID)
            step = np.max([int(STEPRATIO*nx.number_of_nodes(g_kcore)),1]) #step size
            inner_env.s0(g_kcore_inner)
            ##do dqn attack and idMapping
            list_pred = self.Predict([g_kcore_inner], [inner_env.action_list], False)
            batchSol = np.argsort(-list_pred[0])[:step]
            for node in batchSol:
                OriginNodeID = ID2Node[node]
                sol_decycle.append(int(OriginNodeID))

            #################### update graph status ######################
            sol_len = 0
            while sol_len < len(batchSol) and len(g_kcore):
                if max_score_dict[1] != None:   # 如果g_kcore中某些节点的度小于2，则先移除这些节点
                    d = 1
                    mx_scr_d = max_score_dict[d]
                    v = random.choice(list(H[d][mx_scr_d].keys()))
                else:
                    v = ID2Node[batchSol[sol_len]]
                    sol_len += 1
                    if v in g_kcore.nodes():
                        d = max(g_kcore.degree(v), 1)
                        mx_scr_d = max_score_dict[d]
                    else:
                        continue

                if v in g_kcore.nodes():
                    # remove element from H and set its score to None
                    del H[d][mx_scr_d][v]
                    score_dict[v] = None

                    # check for newest largest score
                    if H[d][mx_scr_d] == {}:	#判断度数为d的节点集合是否为空
                        del H[d][mx_scr_d]
                    # update max_score_dict
                    # suboptimal
                    try:
                        max_score_dict[d] = max(H[d].keys())
                    except ValueError:
                        max_score_dict[d] = None
                    ## update the neighbors
                    dv = g_kcore[v]
                    # remove neighbors from dict (for now)
                    for nb in dv:
                        self.remove_node_by_score(nb,g_kcore,H,max_score_dict,score_dict,2)
                    ## remove v from G
                    g_kcore.remove_node(v)
                    # update the degrees of the neighbors and their scores
                    for nb in dv:
                        self.add_node_by_score(nb,g_kcore,H,max_score_dict,score_dict,2)
                    ## check if dmax needs updating and do so if necessary. not necessary
                    if max_score_dict[d] == None:
                        # attempt to update
                        try:
                            while max_score_dict[dmax] == None:
                                dmax -= 1
                        # unless there are no nodes left
                        except KeyError:
                            done = True
                            #sys.exit('Error!')
            ##################################################################
            N = len(g_kcore)

        sol_treebreak = TreeBreaker.TreeBreak(g_original, sol_decycle)
        sol = sol_decycle + sol_treebreak
        solution = sol
        for i in range(len(solution)):
            f_out.write('%d\n' % int(solution[i]))
            f_out.flush()
        f_out.close()